In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
pd.set_option('display.expand_frame_repr', False)

In [ ]:
data = pd.read_csv('cr_loan2.csv')
print(data)

In [ ]:
data["loan_status"].value_counts().plot(kind='bar')
plt.title("Loan Status Distribution")
plt.xlabel("Loan Status")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

In [ ]:
print(data.describe())

In [ ]:
data = data.drop(data[data["person_age"] > 100].index)
indices = data[data["person_emp_length"] > 60].index
data = data.drop(indices)

In [ ]:
data.isna().sum()

In [ ]:
data['person_emp_length'] = data['person_emp_length'].fillna(data['person_emp_length'].median())
indices = data[data['loan_int_rate'].isnull()].index
data = data.drop(indices)

In [ ]:
data_num = data.select_dtypes(exclude=['object'])
data_str = data.select_dtypes(include=['object'])
data_str_onehot = pd.get_dummies(data_str)
data = pd.concat([data_num, data_str_onehot], axis=1)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay, auc, roc_curve
from sklearn.metrics import classification_report
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import KFold
import xgboost as xgb

In [ ]:
X = data.drop('loan_status', axis=1)
y = data[['loan_status']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=2026)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_slr = scaler.fit_transform(X_train) 
X_test_slr = scaler.transform(X_test)

In [ ]:
log_clf = LogisticRegression(solver='lbfgs', max_iter=1000)
rnd_clf = RandomForestClassifier()
svm_clf = SVC()
knn_clf = KNeighborsClassifier()
grd_clf = GradientBoostingClassifier()
xgb_clf = xgb.XGBClassifier()

In [ ]:
for clf in (log_clf, rnd_clf, svm_clf, knn_clf, grd_clf, xgb_clf):
    clf.fit(X_train_slr, np.ravel(y_train))
    y_pred = clf.predict(X_test_slr)
    print(f"The {clf.__class__.__name__} has an Accuracy: {accuracy_score(y_test, y_pred):.4f} , and a Precision of: {precision_score(y_test, y_pred):.4f} ")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
class_names = ['Non-Default', 'Default']
display_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
display_cm.plot()
plt.show()